# Recorte de enunciados de PDFs de emestrada — paso a paso

In [56]:
# %pip install pdfplumber pypdf pandas

import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
from pypdf import PdfReader, PdfWriter, Transformation
from IPython.display import display

Establecer directorio raiz

In [57]:
try:
    root = "/home/daniel/code/clasificador_selectividad"
    os.chdir(root)
except Exception as e:
    root = "/Users/daniel/code/clasificador_selectividad"
    os.chdir(root)

Parámetros para medidas de recuadros y pdf

In [58]:
ANCHO_MIN, ALTO_MIN = 300, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

Función para detectar los recuadros

In [59]:
def detect_rectangles(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [60]:
# PATRON = re.compile(
#     r"""
#     (?P<asig>SOCIALES[ \t]+(?:II|2)|MATEM[AÁ]TICAS[ \t]+(?:II|2)|QU[IÍ]MICA|F[IÍ]SICA)
#     [ \t.,:;-]*
#     (?P<anio>(?:19|20)\d{2})
#     [ \t.,:;-]*
#     (?P<conv>(?:JUNIO|JULIO|SEPTIEMBRE|PONENCIA|RESERVA|MODELO)(?:[ \t]*\d+)?)
#     [ \t.,:;-]*
#     (?P<ej>[^\n\r]*?)
#     [ \t.]*\r?$
#     """,
#     re.IGNORECASE | re.MULTILINE | re.VERBOSE,
# )

Extraer y procesar los datos del examen

In [61]:
def process_exercise_info(exercise_info):
    exercise_info = exercise_info.lower()
    asignaturas = {
        "química" : "Química",
        "quimica" : "Química",
        "física" : "Física",
        "fisica" : "Física",
        "sociales ii" : "Mates CCSS",
        "matemáticas ii" : "Matemáticas II",
        "matematicas ii" : "Matemáticas II"
    }

    # extraer asignatura
    for k,v in asignaturas.items():
        if k in exercise_info:
            asignatura = v
            break
        else:
            asignatura = "asignatura_desconocida"

    # extraer año
    year_pattern = r"2\d{3}"
    year = re.findall(year_pattern, exercise_info)[0]

    # extraer convocatoria y ejercicio
    convocatorias = ["junio", "julio", "septiembre",
                     "modelo", "ponencia",
                     "reserva"]

    # convocatorias de Junio, Julio, Septiembre, modelo o ponencia
    for conv in convocatorias[:-2]:
        if conv in exercise_info:
            convocatoria = conv.title()
            convocatoria_position = exercise_info.find(conv)
            ejercicio = (" ".join(exercise_info[convocatoria_position:].split(" ")[1:])
                         .title()
            )
            break

    # convocatoria es una reserva
    if "reserva" in exercise_info:
        convocatoria_position = exercise_info.find("reserva")
        convocatoria = (" ".join(exercise_info[convocatoria_position:]
                                 .split(" ")[:2])
                                 .title()
                                 .replace(".", "")
        )
        ejercicio = (" ".join(
            exercise_info[convocatoria_position:].split(" ")[2:]
            ).title()
        )

    return (asignatura, year, convocatoria, ejercicio)   


Limpieza de texto

In [62]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

Procesamiento de un archivo pdf con enunciados

In [ ]:
def procesar_pdf(ruta_pdf, output_folder, export_pdf = True, export_png=True, dpi=DPI, uno_por_pagina=False):
    exercises, clips = [], []
    png_folder = output_folder / "enunciados_PNG" / f"{ruta_pdf.stem}_png"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        # procesamiento de las páginas del pdf
        for n, page in enumerate(pdf.pages):
            # detectar los recuadros
            boxes = detect_rectangles(page)

            # obtener coordenadas del recuadro
            for k, r in enumerate(boxes, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                # recortar el trozo del recuadro del pdf
                clip = page.crop((x0, top, x1, bottom))

                # extraer el texto del enunciado en
                # el recuadro
                texto_raw = clip.extract_text() or ""

                # separar el enunciado en líneas
                lineas = texto_raw.split("\n")

                # el enunciado son todas las líneas excepto
                # la última (información del ejercicio)
                texto = "\n".join(lineas[:-1])

                # información del ejercicio
                exercise_info = lineas[-1]
                asignatura, anio, conv, ej = process_exercise_info(exercise_info)
                # texto = texto_raw.replace("\n", " ")
                # metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                # crear nombre de archivo png
                # a partir de la información del ejercicio
                if exercise_info:
                    ej_slug = slug(ej)
                    png_name = f"{asignatura}_{anio}_{slug(conv)}_{ej_slug}"
                    # ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    # nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    png_name = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                # exportar recorte a png
                image = None
                if export_png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    output_file = png_folder / f"{png_name}.png"
                    i = 2
                    while output_file.exists():
                        output_file = png_folder / f"{png_name}_{i}.png"
                        i += 1
                    clip.to_image(resolution=dpi).save(output_file)
                    image = str(output_file.relative_to(output_folder))

                off_x, off_y = float(page.bbox[0]), float(page.bbox[1])
                clips.append((n, off_x + x0, off_y + page.height - bottom,
                                 off_x + x1, off_y + page.height - top))

                aviso = ""
                if not exercise_info:
                    aviso = "sin info ejercicio"
                elif len(boxes) > 1:
                    aviso = f"{len(boxes)} recuadros en la página"

                # asignatura y tema
                #asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]

                exercises.append({
                    "asignatura": asignatura,
                    "asignatura_carpeta" : ruta_pdf.parts[-3].replace(" - Temas", ""),
                    "año" : anio,
                    "convocatoria" : conv,
                    "ejercicio" : ej,
                    "tema": tema,
                    "info_ejercicio" : exercise_info,
                    "enunciado": texto,
                    "archivo": ruta_pdf.name,
                    "imagen": image,
                    "aviso": aviso,
                    #"año": metadata["anio"] if metadata else "",
                    #"convocatoria": metadata["conv"].title() if metadata else "",
                    #"ejercicio": metadata["ej"] if metadata else "",
                })

    if not clips:
        return pd.DataFrame(), "ningún recuadro detectado"
    
    # combinar todos los enunciados en un pdf
    if export_pdf:
        # 2) Composición vectorial con pypdf
        # comprobar si existe el directorio de salida y crearlo si no
        (output_folder / "enunciados_PDF").mkdir(parents=True, exist_ok = True)
        output_pdf_filename = output_folder / "enunciados_PDF" / f"{ruta_pdf.stem}_enunciados.pdf"
        writer = PdfWriter()
        hoja, y, num_hoja = None, None, 0
        already_processed = set()
        reader = PdfReader(ruta_pdf)

        for ex, (n, x0, y0, x1, y1) in zip(exercises, clips):
            # si la página ya se usó (varios recuadros), se relee para no pisar el recorte anterior
            src = (PdfReader(ruta_pdf) if n in already_processed else reader).pages[n]
            already_processed.add(n)
            src.mediabox.lower_left = src.cropbox.lower_left = (x0, y0)
            src.mediabox.upper_right = src.cropbox.upper_right = (x1, y1)
            w, h = x1 - x0, y1 - y0

            if uno_por_pagina:
                hoja = writer.add_blank_page(w, h)
                hoja.merge_transformed_page(src, Transformation().translate(-x0, -y0))
                num_hoja += 1
            else:
                escala = min(1.0, (A4[0] - 2 * MARGEN_A4) / w)
                w, h = w * escala, h * escala
                if hoja is None or y - h < MARGEN_A4:
                    hoja = writer.add_blank_page(*A4)
                    y = A4[1] - MARGEN_A4
                    num_hoja += 1
                tx, ty = (A4[0] - w) / 2, y - h
                t = Transformation().translate(-x0, -y0).scale(escala, escala).translate(tx, ty)
                hoja.merge_transformed_page(src, t)
                y -= h + SEPARACION

            # fila["pdf_salida"] = salida.name
            # fila["pagina_salida"] = num_hoja

        # guardar en pdf los enunciados
        with open(output_pdf_filename, "wb") as f:
            writer.write(f)

    return pd.DataFrame(exercises), ""

## Procesar carpeta

In [66]:
def process_folder(folder, export_pdf = True, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:])
    output_folder = Path(f"{root}/solo_enunciados_ejercicios/{output_folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    # obtener rutas de los archivos pdf de la carpeta
    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    
    # hay límite de archivos por procesar
    if files_to_process_limit:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    dataframes, issues = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            df, aviso = procesar_pdf(file_path, output_folder, export_pdf = export_pdf)
        except Exception as e:
            subject = file_path.parts[-3].replace(" - Temas", "")
            issues.append({"issue": f"ERROR: {e}",
                           "subject" : subject,
                           "pdf file": str(file_path.name)})
            print(f"[{index}/{len(pdf_files)}] ERROR  ({subject}) {file_path.name}: {e}")
            continue

        dataframes.append(df)

        if aviso:
            issues.append({"issue": aviso,
                           "pdf file": str(file_path)})
            
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(df)} ejercicios procesados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame()
    for d in dataframes:
        enunciados_df = pd.concat([enunciados_df, d])
            
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if issues:
        issues_df = pd.DataFrame(issues)
        issues_df.to_csv(output_folder / f"{output_folder.name}_issues.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder}\n\n")

## Procesar carpetas

Normalizar texto de nombres de temas y carpetas

In [67]:
def normalize_text(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

Obtener las rutas de los archivos

In [68]:
os.chdir(f"{root}/source_pdf_files")

In [69]:
carpetas_asignaturas = {d.name:[] for d in Path(".").iterdir() if d.is_dir()}

In [70]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.name for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [71]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

# PROCESAR LOS ARCHIVOS

In [72]:
# process_folder(folder_paths[0], files_to_process_limit=None)

In [73]:
for folder in folder_paths:
    folder_path = Path(folder)
    # Limitar a 5 archivos por carpeta para pruebas
    process_folder(folder, export_pdf = True, files_to_process_limit=2)

Procesando carpeta: Física - Temas/campo_eléctrico_magnético
2 PDFs a procesar

[1/2] OK     2016 - Campo eléctrico y magnético.pdf: 10 ejercicios procesados
[2/2] OK     2017 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
****************************************

Procesamiento completado. Resultados guardados en: /Users/daniel/code/clasificador_selectividad/solo_enunciados_ejercicios/Física - Temas/campo_eléctrico_magnético


Procesando carpeta: Física - Temas/física_cuántica_nuclear
2 PDFs a procesar

[1/2] OK     2016 - Física cuántica y nuclear.pdf: 7 ejercicios procesados
[2/2] OK     2017 - Física cuántica y nuclear.pdf: 12 ejercicios procesados
****************************************

Procesamiento completado. Resultados guardados en: /Users/daniel/code/clasificador_selectividad/solo_enunciados_ejercicios/Física - Temas/física_cuántica_nuclear


Procesando carpeta: Física - Temas/campo_gravitatorio
2 PDFs a procesar

[1/2] OK     2016 - Campo g

## Mover archivos .csv

In [74]:
os.chdir(root)
Path("csv_enunciados").mkdir(exist_ok=True)
os.chdir("solo_enunciados_ejercicios")

In [75]:
for folder in Path(".").iterdir():
    if folder.is_dir():
        subject = folder.name.split(" - ")[0].replace(" ", "_").lower()
        csv_files = list(folder.rglob("*.csv"))
        for csv_file in csv_files:
            new_filename = f"{subject}_{csv_file.stem}.csv"
            if "_issues" in csv_file.name:
                Path(f"{root}/csv_enunciados/issues").mkdir(exist_ok=True)
                output_csv_path = Path(f"{root}/csv_enunciados/issues/{subject}_{csv_file.stem}_issues.csv")
            else:
                new_filename = f"{subject}_{csv_file.stem}_enunciados.csv"
                output_csv_path = Path(f"{root}/csv_enunciados/{new_filename}")

            shutil.move(csv_file, output_csv_path)

Combinar todos los issues

In [76]:
issue_files_path = Path(root) / "csv_enunciados" / "issues"
issue_files = list(issue_files_path.iterdir())
if issue_files:
    issue_df = pd.DataFrame()
    for f in issue_files:
        if f.suffix == ".csv":
            df = pd.read_csv(f, encoding="utf-8")
            issue_df = pd.concat([issue_df, df])
    issue_df.to_csv(issue_files_path /  "../issues.csv", index = False )

# clean issue files
for f in issue_files:
    f.unlink()

# borrar directorio issues
shutil.rmtree(f"{root}/csv_enunciados/issues")

In [77]:
issue_df

,issue,subject,pdf file
0,ERROR: cannot access local variable 'convocato...,Mates II,2024 - Probabilidad.pdf
0,ERROR: cannot access local variable 'convocato...,Mates CCSS,2009 - Contraste De Hipótesis.pdf
0,ERROR: list index out of range,Mates CCSS,MCCSS-T2-2024.pdf
0,ERROR: list index out of range,Química,2001 - Equilibrio Químico.pdf
0,ERROR: cannot access local variable 'convocato...,Mates II,2024 - Distribución normal y binomial.pdf


Combinar en un único CSV

In [78]:
os.chdir(f"{root}/csv_enunciados")

In [79]:
csv_files = [i for i in Path(".").rglob("*.csv") if i.name != "issues.csv"]

In [80]:
enunciados_df = pd.DataFrame()
for f in csv_files:
    df = pd.read_csv(f, encoding="utf-8")
    enunciados_df = pd.concat([enunciados_df, df])
enunciados_df.to_csv("todos_enunciados.csv", encoding="utf-8", index = False)

for f in csv_files:
    f.unlink()

In [51]:
todos_df = pd.read_csv("todos_enunciados.csv")
todos_df.sample(10)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
49,Química,Química,2001,Reserva 1,Ejercicio 4. Opción B,Ácido Base,QUÍMICA. 2001. RESERVA 1. EJERCICIO 4. OPCIÓN B,a) ¿Cuál es la concentración en HNO de una dis...,2001 - Ácido Base.pdf,enunciados_PNG/2001 - Ácido Base_png/Química_...,NaN
328,Física,Física,2016,Reserva 4,Ejercicio 3. Opción A,Campo eléctrico y magnético,FISICA. 2016. RESERVA 4. EJERCICIO 3. OPCIÓN A,"g = 9,8 m s-2\nUna partícula alfa, con una ene...",2016 - Campo eléctrico y magnético.pdf,enunciados_PNG/2016 - Campo eléctrico y magne...,NaN
315,asignatura_desconocida,Mates II,2001,Reserva 1,Ejercicio 3. Opción A.,Matrices Y Determinantes,M ATEMÁTICAS II. 2001. RESERVA 1. EJERCICIO 3....,1 2 3\n1 2 1 2 3 1 1  \nD e las matr...,2001 - Matrices Y Determinantes.pdf,enunciados_PNG/2001 - Matrices Y Determinantes...,NaN
101,Física,Física,2016,Septiembre,Ejercicio 2. Opción B,Ondas,FISICA. 2016. SEPTIEMBRE. EJERCICIO 2. OPCIÓN B,a) Periodicidad espacial y temporal de las ond...,2016 - Ondas.pdf,enunciados_PNG/2016 - Ondas_png/Física_2016_se...,NaN
32,asignatura_desconocida,Mates CCSS,2002,Reserva 4,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2002. RESERVA 4. EJERCICIO 3. PA...,S e dispone de una baraja española de 40 carta...,2002 - Probabilidad.pdf,enunciados_PNG/2002 - Probabilidad_png/asignat...,NaN
179,Matemáticas II,Mates II,2001,Reserva 2,Ejercicio 4. Opción B.,Geometría,MATEMÁTICAS II. 2001. RESERVA 2. EJERCICIO 4. ...,3x2y 0\nSea r la recta de ecuaciones r  ....,2001 - Geometría.pdf,enunciados_PNG/2001 - Geometría_png/Matemátic...,NaN
277,Química,Química,2000,Septiembre,Ejercicio 1. Opción A,Formulación,QUÍMICA. 2000. SEPTIEMBRE. EJERCICIO 1. OPCIÓN A,Formule o nombre los compuestos siguientes: a)...,2000 - Formulación.pdf,enunciados_PNG/2000 - Formulación_png/Química...,NaN
354,Química,Química,2001,Septiembre,Ejercicio 4. Opción A,Reactividad Orgánica,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 4. OPCIÓN A,Razone si son verdaderas o falsas las siguient...,2001 - Reactividad Orgánica.pdf,enunciados_PNG/2001 - Reactividad Orgánica_pn...,NaN
245,Matemáticas II,Mates II,2000,Junio,Ejercicio 1. Opción A.,Integrales,MATEMÁTICAS II. 2000. JUNIO. EJERCICIO 1. OPCI...,a) Dibuja el recinto limitado por las curvas: ...,2000 - Integrales.pdf,enunciados_PNG/2000 - Integrales_png/Matemátic...,NaN
302,Química,Química,2001,Reserva 3,Ejercicio 3. Opción A,Reacciones Redox,QUÍMICA. 2001. RESERVA 3. EJERCICIO 3. OPCIÓN A,Dadas las siguientes reacciones (sin ajustar):...,2001 - Reacciones Redox.pdf,enunciados_PNG/2001 - Reacciones Redox_png/Quí...,NaN


In [58]:
todos_df.asignatura.value_counts()

asignatura
Química                   139
Física                     88
Matemáticas II             85
Mates CCSS                 74
asignatura_desconocida     27
Name: count, dtype: int64

In [52]:
todos_df[todos_df.asignatura == "asignatura_desconocida"]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
14,asignatura_desconocida,Mates CCSS,2001,Reserva 1,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2001 RESERVA 1. EJERCICIO 3. PAR...,2\nDado un espacio muestral E se consideran lo...,2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
15,asignatura_desconocida,Mates CCSS,2001,Reserva 1,Ejercicio 3. Parte I. Opción B,Probabilidad,S OCIALES II. 2001 RESERVA 1. EJERCICIO 3. PAR...,E l 35 % de los estudiantes de un centro docen...,2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
16,asignatura_desconocida,Mates CCSS,2001,Reserva 2,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2001. RESERVA 2. EJERCICIO 3. PA...,"D os cajas, A y B, tienen el siguiente conteni...",2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
19,asignatura_desconocida,Mates CCSS,2001,Reserva 3,Ejercicio 3. Parte I. Opción B,Probabilidad,S OCIALES II. 2001. RESERVA 3. EJERCICIO 3. PA...,E n un cineclub hay 80 películas; 60 son de “a...,2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
20,asignatura_desconocida,Mates CCSS,2001,Reserva 4,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2001. RESERVA 4. EJERCICIO 3. PA...,E n una ciudad el 60 % de sus habitantes son a...,2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
21,asignatura_desconocida,Mates CCSS,2001,Reserva 4,Ejercicio 3. Parte I. Opción B,Probabilidad,S OCIALES II. 2001 RESERVA 4. EJERCICIO 3. PAR...,T enemos un cofre A con 2 monedas de oro y 3 d...,2001 - Probabilidad.pdf,enunciados_PNG/2001 - Probabilidad_png/asignat...,NaN
25,asignatura_desconocida,Mates CCSS,2002,Junio,Ejercicio 3. Parte I. Opción B,Probabilidad,S OCIALES II. 2002. JUNIO. EJERCICIO 3. PARTE ...,S egún la estadística de los resultados en las...,2002 - Probabilidad.pdf,enunciados_PNG/2002 - Probabilidad_png/asignat...,NaN
26,asignatura_desconocida,Mates CCSS,2002,Reserva 1,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2002 RESERVA 1. EJERCICIO 3. PAR...,"U na urna contiene 15 bolas, de las cuales 6 s...",2002 - Probabilidad.pdf,enunciados_PNG/2002 - Probabilidad_png/asignat...,NaN
27,asignatura_desconocida,Mates CCSS,2002,Reserva 1,Ejercicio 3. Parte I. Opción B,Probabilidad,S OCIALES II. 2002 RESERVA 1. EJERCICIO 3. PAR...,"T enemos 3 estuches de lápices A, B y C. El es...",2002 - Probabilidad.pdf,enunciados_PNG/2002 - Probabilidad_png/asignat...,NaN
28,asignatura_desconocida,Mates CCSS,2002,Reserva 2,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2002. RESERVA 2. EJERCICIO 3. PA...,"E l despertador de Pedro no funciona bien, pue...",2002 - Probabilidad.pdf,enunciados_PNG/2002 - Probabilidad_png/asignat...,NaN


In [55]:
todos_df.loc[315,:]

asignatura                                       asignatura_desconocida
asignatura_carpeta                                             Mates II
año                                                                2001
convocatoria                                                  Reserva 1
ejercicio                                        Ejercicio 3. Opción A.
tema                                           Matrices Y Determinantes
info_ejercicio        M ATEMÁTICAS II. 2001. RESERVA 1. EJERCICIO 3....
enunciado             1 2 3\n1 2 1 2 3 1 1  \nD e las matr...
archivo                             2001 - Matrices Y Determinantes.pdf
imagen                enunciados_PNG/2001 - Matrices Y Determinantes...
aviso                                                               NaN
Name: 315, dtype: object